In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import os
import time

In [2]:
# Assume the project root is the parent of src/
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [3]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "JNJ", "JPM", "XOM", "CAT", "PG", "NEE"]
ticker_data = pd.DataFrame()


for ticker in tickers:
    data = yf.download(ticker, start='2020-01-01', end='2025-01-01', auto_adjust=True)
    # Only keep Adjusted Close
    features = data[['Close','Volume']].copy()
    features.columns = [f"{ticker}_Close", f"{ticker}_Volume"]
    ticker_data = pd.concat([ticker_data, features], axis=1)
    time.sleep(1)  # sleep to avoid hitting API limits

ticker_data.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,AAPL_Close,AAPL_Volume,MSFT_Close,MSFT_Volume,NVDA_Close,NVDA_Volume,AMZN_Close,AMZN_Volume,JNJ_Close,JNJ_Volume,JPM_Close,JPM_Volume,XOM_Close,XOM_Volume,CAT_Close,CAT_Volume,PG_Close,PG_Volume,NEE_Close,NEE_Volume
Date,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.538513,135480400,152.791153,22622100,5.971410,237536000,94.900497,80580000,124.072983,5777000,119.573372,10803700,54.131073,12456400,132.874985,3311900,106.273247,8130800,51.782585,7884800
2020-01-03,71.833298,146322800,150.888611,21116200,5.875832,205384000,93.748497,75288000,122.636490,5752400,117.995430,10386800,53.695885,17386900,131.030060,3100600,105.558487,7970500,52.151508,7097200
2020-01-06,72.405678,118387200,151.278641,20813700,5.900473,262636000,95.143997,81236000,122.483498,7731300,117.901604,10259000,54.108177,20081900,130.941803,2549600,105.704895,6674400,52.411919,5518800
2020-01-07,72.065140,108872000,149.899292,21634100,5.971909,314856000,95.343002,80898000,123.231468,7382900,115.897224,10531300,53.665344,17387700,129.211655,2841900,105.050415,7583400,52.366341,6653200
2020-01-08,73.224411,132079200,152.286972,27746500,5.983109,277108000,94.598503,70160000,123.214493,6605800,116.801315,9695300,52.856052,15137700,130.359177,2153200,105.498222,5385100,52.342468,5936000


In [4]:
returns = ticker_data[[col for col in ticker_data.columns if "Close" in col]].pct_change().dropna()
volume = ticker_data[[col for col in ticker_data.columns if "Volume" in col]].iloc[1:]
ml_data = pd.concat([returns, volume], axis=1)
print(ml_data.head())

            AAPL_Close  MSFT_Close  NVDA_Close  AMZN_Close  JNJ_Close  \
Date                                                                    
2020-01-03   -0.009722   -0.012452   -0.016006   -0.012139  -0.011578   
2020-01-06    0.007968    0.002585    0.004194    0.014886  -0.001248   
2020-01-07   -0.004703   -0.009118    0.012107    0.002092   0.006107   
2020-01-08    0.016086    0.015929    0.001876   -0.007809  -0.000138   
2020-01-09    0.021241    0.012493    0.010983    0.004799   0.002966   

            JPM_Close  XOM_Close  CAT_Close  PG_Close  NEE_Close  AAPL_Volume  \
Date                                                                            
2020-01-03  -0.013196  -0.008040  -0.013885 -0.006726   0.007124    146322800   
2020-01-06  -0.000795   0.007678  -0.000674  0.001387   0.004993    118387200   
2020-01-07  -0.017000  -0.008184  -0.013213 -0.006192  -0.000870    108872000   
2020-01-08   0.007801  -0.015080   0.008881  0.004263  -0.000456    132079200   
20

In [5]:
path = os.path.join(data_dir,"ML_DATA.csv")
ml_data.to_csv(path)